In [1]:
from dataclasses import asdict
from datetime import datetime
from typing import Optional, List, Dict

In [2]:
import firebase_admin
from firebase_admin import credentials, firestore

SERVICE_ACCOUNT_PATH = "firebase/serviceAccountKey.json.json"

if not firebase_admin._apps:
    cred = credentials.Certificate(SERVICE_ACCOUNT_PATH)
    firebase_admin.initialize_app(cred)

db = firestore.client()

print("Firestore repository connection ready!")

Firestore repository connection ready!


In [3]:
class PatientRepository:

    def __init__(self, db):
        self.db = db

    # -------------------------
    # PATIENT
    # -------------------------

    def save_patient(self, patient_id: str, data: dict):
        self.db.collection("patients").document(patient_id).set(
            data,
            merge=True
        )

    def get_patient(self, patient_id: str) -> Optional[dict]:
        doc = self.db.collection("patients").document(patient_id).get()

        if not doc.exists:
            return None

        return doc.to_dict()

    # -------------------------
    # MEMORIES
    # -------------------------

    def save_memory(self, memory_id: str, data: dict):
        self.db.collection("patient_memories").document(memory_id).set(
            data,
            merge=True
        )

    def get_memory(self, memory_id: str) -> Optional[dict]:
        doc = self.db.collection("patient_memories").document(memory_id).get()

        if not doc.exists:
            return None

        return doc.to_dict()

    def get_patient_memories(self, patient_id: str) -> List[dict]:

        docs = (
            self.db.collection("patient_memories")
            .where("patient_id", "==", patient_id)
            .stream()
        )

        return [doc.to_dict() for doc in docs]

    # -------------------------
    # ROUTINES
    # -------------------------

    def save_routine(self, routine_id: str, data: dict):
        self.db.collection("routines").document(routine_id).set(
            data,
            merge=True
        )

    def get_patient_routines(self, patient_id: str) -> List[dict]:

        docs = (
            self.db.collection("routines")
            .where("patient_id", "==", patient_id)
            .stream()
        )

        return [doc.to_dict() for doc in docs]

    # -------------------------
    # GOALS
    # -------------------------

    def save_goal(self, goal_id: str, data: dict):
        self.db.collection("goals").document(goal_id).set(
            data,
            merge=True
        )

    def get_patient_goals(self, patient_id: str) -> List[dict]:

        docs = (
            self.db.collection("goals")
            .where("patient_id", "==", patient_id)
            .stream()
        )

        return [doc.to_dict() for doc in docs]

    # -------------------------
    # SESSION
    # -------------------------

    def save_session(self, session_id: str, data: dict):

        data["updated_at"] = datetime.utcnow().isoformat()

        self.db.collection("sessions").document(session_id).set(
            data,
            merge=True
        )

    def get_session(self, session_id: str) -> Optional[dict]:

        doc = self.db.collection("sessions").document(session_id).get()

        if not doc.exists:
            return None

        return doc.to_dict()

    # -------------------------
    # ACTIVITY RESULTS
    # -------------------------

    def save_activity_result(self, result_id: str, data: dict):

        data["created_at"] = datetime.utcnow().isoformat()

        self.db.collection("activity_results").document(result_id).set(
            data,
            merge=True
        )

In [4]:
repository = PatientRepository(db)

print("PatientRepository created successfully!")

PatientRepository created successfully!


In [5]:
patient = {
    "patient_id": "lakshmi_001",
    "name": "Lakshmi",

    "preferred_language": "te",

    "supported_languages": [
        "te",
        "en",
        "hi",
        "as"
    ],

    "interaction_mode": "voice_touch",

    "voice_enabled": True,
    "voice_input_enabled": True,
    "voice_output_enabled": True,

    "status": "active"
}

repository.save_patient(
    "lakshmi_001",
    patient
)

print("Patient saved!")

Patient saved!


In [12]:
memories = [
    {
        "memory_id": "mem_001",
        "patient_id": "lakshmi_001",
        "title": "Lakshmi's daughter Anu",
        "description": "Anu is Lakshmi's daughter.",
        "people": ["Anu"],
        "places": [],
        "objects": [],
        "events": [],
        "emotions": ["happy"],
        "tags": ["family", "daughter", "anu"],
        "language": "en",
        "consent": True
    },
    {
        "memory_id": "mem_002",
        "patient_id": "lakshmi_001",
        "title": "Lakshmi's kitchen",
        "description": "Lakshmi often spends time in her kitchen.",
        "people": [],
        "places": ["home", "kitchen"],
        "objects": [],
        "events": ["cooking"],
        "emotions": [],
        "tags": ["home", "kitchen", "daily_life"],
        "language": "en",
        "consent": True
    },
    {
        "memory_id": "mem_003",
        "patient_id": "lakshmi_001",
        "title": "Making chapathi",
        "description": "Lakshmi enjoys making chapathi with her daughter Anu.",
        "people": ["Anu"],
        "places": ["kitchen"],
        "objects": ["chapathi"],
        "events": ["cooking"],
        "emotions": ["happy"],
        "tags": ["family", "daughter", "cooking", "chapathi"],
        "language": "en",
        "consent": True
    }
]

for memory in memories:
    repository.save_memory(
        memory["memory_id"],
        memory
    )

print("Memories saved successfully!")

Memories saved successfully!


In [13]:
patient = repository.get_patient("lakshmi_001")

print(patient)

{'name': 'Lakshmi', 'patient_id': 'lakshmi_001', 'voice_output_enabled': True, 'status': 'active', 'voice_enabled': True, 'interaction_mode': 'voice_touch', 'preferred_language': 'te', 'voice_input_enabled': True, 'supported_languages': ['te', 'en', 'hi', 'as']}


In [16]:
memories = repository.get_patient_memories("lakshmi_001")

print("Number of memories:", len(memories))

for memory in memories:
    print("\n", memory["memory_id"])
    print("Title:", memory["title"])
    print("Description:", memory["description"])

Number of memories: 3

 mem_001
Title: Lakshmi's daughter Anu
Description: Anu is Lakshmi's daughter.

 mem_002
Title: Lakshmi's kitchen
Description: Lakshmi often spends time in her kitchen.

 mem_003
Title: Making chapathi
Description: Lakshmi enjoys making chapathi with her daughter Anu.


In [17]:
routines = [
    {
        "routine_id": "water_0830",
        "patient_id": "lakshmi_001",
        "routine_type": "hydration",
        "title": "Drink water",
        "scheduled_time": "08:30",
        "description": "Drink water after breakfast.",
        "enabled": True
    },
    {
        "routine_id": "water_1030",
        "patient_id": "lakshmi_001",
        "routine_type": "hydration",
        "title": "Drink water",
        "scheduled_time": "10:30",
        "description": "Drink water during the morning.",
        "enabled": True
    },
    {
        "routine_id": "water_1300",
        "patient_id": "lakshmi_001",
        "routine_type": "hydration",
        "title": "Drink water",
        "scheduled_time": "13:00",
        "description": "Drink water around lunchtime.",
        "enabled": True
    },
    {
        "routine_id": "water_1530",
        "patient_id": "lakshmi_001",
        "routine_type": "hydration",
        "title": "Drink water",
        "scheduled_time": "15:30",
        "description": "Drink water during the afternoon.",
        "enabled": True
    },
    {
        "routine_id": "water_1800",
        "patient_id": "lakshmi_001",
        "routine_type": "hydration",
        "title": "Drink water",
        "scheduled_time": "18:00",
        "description": "Drink water in the evening.",
        "enabled": True
    }
]

for routine in routines:
    repository.save_routine(
        routine["routine_id"],
        routine
    )

print("Routines saved successfully!")

Routines saved successfully!


In [18]:
routines = repository.get_patient_routines("lakshmi_001")

print("Number of routines:", len(routines))

for routine in routines:
    print(
        routine["routine_id"],
        "|",
        routine["scheduled_time"],
        "|",
        routine["title"]
    )

Number of routines: 5
water_0830 | 08:30 | Drink water
water_1030 | 10:30 | Drink water
water_1300 | 13:00 | Drink water
water_1530 | 15:30 | Drink water
water_1800 | 18:00 | Drink water


In [19]:
print("=" * 50)
print("PATIENT REPOSITORY")
print("=" * 50)

patient = repository.get_patient("lakshmi_001")
memories = repository.get_patient_memories("lakshmi_001")
routines = repository.get_patient_routines("lakshmi_001")
goals = repository.get_patient_goals("lakshmi_001")

print("Patient:", patient["name"])
print("Language:", patient["preferred_language"])
print("Memories:", len(memories))
print("Routines:", len(routines))
print("Goals:", len(goals))

PATIENT REPOSITORY
Patient: Lakshmi
Language: te
Memories: 3
Routines: 5
Goals: 1


In [9]:
goal = {
    "goal_id": "goal_001",
    "patient_id": "lakshmi_001",
    "goal_type": "routine_recall",
    "priority": 10,
    "description": "Help the patient remember daily routines.",
    "source": "caregiver",
    "active": True
}

repository.save_goal("goal_001", goal)

print("Goal saved!")

Goal saved!


In [10]:
goals = repository.get_patient_goals("lakshmi_001")

print("Goals:", len(goals))

for goal in goals:
    print(goal)

Goals: 1
{'active': True, 'goal_id': 'goal_001', 'goal_type': 'routine_recall', 'description': 'Help the patient remember daily routines.', 'source': 'caregiver', 'priority': 10, 'patient_id': 'lakshmi_001'}


In [11]:
print("=" * 50)
print("COGNITIVE AI PATIENT REPOSITORY TEST")
print("=" * 50)

patient = repository.get_patient("lakshmi_001")
memories = repository.get_patient_memories("lakshmi_001")
routines = repository.get_patient_routines("lakshmi_001")
goals = repository.get_patient_goals("lakshmi_001")

print("Patient:", patient["name"])
print("Preferred language:", patient["preferred_language"])
print("Supported languages:", patient["supported_languages"])
print("Memories:", len(memories))
print("Routines:", len(routines))
print("Goals:", len(goals))

print("\nRepository is working!")

COGNITIVE AI PATIENT REPOSITORY TEST
Patient: Lakshmi
Preferred language: te
Supported languages: ['te', 'en', 'hi', 'as']
Memories: 0
Routines: 0
Goals: 1

Repository is working!
